# Multimodal Classification

> Vision-language classification, contrastive learning, and VQA with `MultimodalClassificationLearner`.

This notebook provides practical starting points for CLIP/BLIP/SigLIP-style workflows.

## Notebook Metadata

`nbdev` export target for this notebook.

In [ ]:
#| default_exp multimodal.classification


## Setup

Import dependencies used in the examples.

In [ ]:
import warnings

warnings.filterwarnings('ignore')

import pandas as pd

from blurr.multimodal.classification import MultimodalClassificationLearner


## Quick Start

Create a multimodal classifier from `(image, text)` pairs and labels.

In [ ]:
bootstrap_df = pd.DataFrame({
    'image': [
        'https://images.unsplash.com/photo-1517849845537-4d257902454a',
        'https://images.unsplash.com/photo-1517423440428-a5a00ad493e8',
    ],
    'text': [
        'a photo of a cat on a couch',
        'a photo of a dog outdoors',
    ],
    'label': ['cat', 'dog'],
})

learn = MultimodalClassificationLearner.from_df(
    bootstrap_df,
    model_name='openai/clip-vit-base-patch32',
    image_col='image',
    text_col='text',
    label_col='label',
    task='classification',
    bs=2,
)

# Optional: train briefly
# learn.fit_one_cycle(1, 2e-4)

quick_pred = learn.blurr_predict((
    'https://images.unsplash.com/photo-1519052537078-e6302a4968d4',
    'a photo of a dog',
))[0]
quick_pred


**Expected output**

A classification-style dictionary (`label`, `score`, `probs`, `class_labels`) for each `(image, text)` input pair.

## Zero-Shot Prediction

Use CLIP/SigLIP-compatible backbones to score candidate labels without supervised fine-tuning.

In [ ]:
zs = learn.blurr_zero_shot_predict(
    'https://images.unsplash.com/photo-1519052537078-e6302a4968d4',
    candidate_labels=['cat', 'dog', 'car'],
    prompt_template='a photo of a {}',
)[0]
zs


## Advanced: Contrastive Mode

Set `task='contrastive'` to optimize paired image-text alignment objectives.

In [ ]:
contrastive_learn = MultimodalClassificationLearner.from_df(
    bootstrap_df,
    model_name='openai/clip-vit-base-patch32',
    image_col='image',
    text_col='text',
    task='contrastive',
    bs=2,
)


## Advanced: VQA Mode

For visual question answering, use `task='vqa'` with question text + answer labels.

In [ ]:
# vqa_df = pd.DataFrame({
#     'image': ['path/to/image1.jpg', 'path/to/image2.jpg'],
#     'text': ['What color is the car?', 'How many dogs are visible?'],
#     'label': ['red', '2'],
# })
#
# vqa_learn = MultimodalClassificationLearner.from_df(
#     vqa_df,
#     model_name='Salesforce/blip-vqa-base',
#     image_col='image',
#     text_col='text',
#     label_col='label',
#     task='vqa',
# )


## API Reference

### Main class
- `MultimodalClassificationLearner`

### Construction methods
- `from_df(df, model_name, task='classification'|'contrastive'|'vqa', ...)`
- `from_hf(dataset, model_name, ...)`

### Inference
- `blurr_predict(items, threshold=0.5)`
- `blurr_zero_shot_predict(images, candidate_labels, prompt_template='a photo of {}')`

### Supported tasks
- `classification` (single- or multi-label)
- `contrastive` (CLIP-style alignment)
- `vqa` (visual question answering)
